## XGBoost

[XGBoost](https://xgboost.readthedocs.io/en/latest/) is a well-known gradient boosted decision trees (GBDT) machine learning package used to tackle regression, classification, and ranking problems. XGBoost now includes seamless, drop-in GPU acceleration, which significantly speeds up model training and improves accuracy for better predictions. 

In this lab, you will learn how to train a XGBoost model easily to make predictions for your business.

Let's start with generating some fake dataset

In [ ]:
from sklearn.datasets import make_classification, make_regression
import numpy as np
import cudf
import xgboost as xgb
# helper function for simulating data
def simulate_data(m, n, k=2, random_state=None, classification=True):
    if classification:
        features, labels = make_classification(m, n_features=n, n_informative=int(n/5),
                                               n_redundant=0, 
                                               n_clusters_per_class=2, 
                                               n_classes=k,
                                               class_sep=0.5, random_state=random_state)
        features = features.astype(np.float32)
    else:
        features, labels = make_regression(n_samples=m, n_features=n, 
                                           n_informative=int(n/5), n_targets=1, 
                                           random_state=random_state)
    
        features = features.astype(np.float32)
        labels = labels.astype(np.float32)
    df = cudf.DataFrame({i:features[:,i] for i in range(n)})
    df['label'] = labels
    return df

Generate a dataframe with 10 features where only 2 features are informative. The target variable is a binary variable indicating true or false state of the event.

In [ ]:
length = 10000
n_cols = 10
data = simulate_data(length, n_cols, random_state=3, classification=True)
data

Split the dataset into training and validation datasets:

In [ ]:
training_frac = 0.8
validation = int(length*training_frac)
train_df = data.iloc[:validation]
val_df = data.iloc[validation:]
dtrain = xgb.DMatrix(train_df[list(range(n_cols))], train_df['label'])
dval = xgb.DMatrix(val_df[list(range(n_cols))], val_df['label'])

We are ready to build the XGBoost model. Let's go over the popular parameter settings the XGBoost first.

## General Parameters

These define the overall functionality of XGBoost.

1. booster [default=gbtree]
    - Select the type of model to run at each iteration. It has 2 options:
        - gbtree: tree-based models
        - gblinear: linear models
2. silent [default=0]:
    - Silent mode is activated is set to 1, i.e. no running messages will be printed.
    - It?s generally good to keep it 0 as the messages might help in understanding the model.
 

## Booster Parameters
Though there are 2 types of boosters, I?ll consider only tree booster here because it always outperforms the linear booster and thus the later is rarely used.

1. eta [default=0.3]
    - Makes the model more robust by shrinking the weights on each step
    - Typical final values to be used: 0.01-0.2
2. min_child_weight [default=1]
    - Defines the minimum sum of weights of all observations required in a child.
3. max_depth [default=6]
    - Used to control over-fitting as higher depth will allow model to learn relations very specific to a particular sample.
    - Should be tuned using CV.
    - Typical values: 3-10
4. max_leaf_nodes
    - The maximum number of terminal nodes or leaves in a tree.
    - Can be defined in place of max_depth. Since binary trees are created, a depth of ?n? would produce a maximum of 2^n leaves.
5. gamma [default=0]
    - A node is split only when the resulting split gives a positive reduction in the loss function. Gamma specifies the minimum loss reduction required to make a split.
    - Makes the algorithm conservative. The values can vary depending on the loss function and should be tuned.
6. subsample [default=1]
    - Denotes the fraction of observations to be randomly samples for each tree.
    - Lower values make the algorithm more conservative and prevents overfitting but too small values might lead to under-fitting.
    - Typical values: 0.5-1
7. colsample_bytree [default=1]
    - Denotes the fraction of columns to be randomly samples for each tree.
    - Typical values: 0.5-1
8. colsample_bylevel [default=1]
    - Denotes the subsample ratio of columns for each split, in each level.
    - I don?t use this often because subsample and colsample_bytree will do the job for you. but you can explore further if you feel so.
9. lambda [default=1]
    - L2 regularization term on weights (analogous to Ridge regression)
    - This used to handle the regularization part of XGBoost. Though many data scientists don?t use it often, it should be explored to reduce overfitting.
10. alpha [default=0]
    - L1 regularization term on weight (analogous to Lasso regression)
    - Can be used in case of very high dimensionality so that the algorithm runs faster when implemented
11. scale_pos_weight [default=1]
    - A value greater than 0 should be used in case of high class imbalance as it helps in faster convergence.
 

## Learning Task Parameters
These parameters are used to define the optimization objective the metric to be calculated at each step.

1. objective [default=reg:linear]
    - This defines the loss function to be minimized. Mostly used values are:
        - binary:logistic ?logistic regression for binary classification, returns predicted probability (not class)
        - multi:softmax ?multiclass classification using the softmax objective, returns predicted class (not probabilities)
            you also need to set an additional num_class (number of classes) parameter defining the number of unique classes
        - multi:softprob ?same as softmax, but returns predicted probability of each data point belonging to each class.
2. eval_metric [ default according to objective ]
    - The metric to be used for validation data.
    - The default values are rmse for regression and error for classification.
    - Typical values are:
        - rmse ? root mean square error
        - mae ? mean absolute error
        - logloss ? negative log-likelihood
        - error ? Binary classification error rate (0.5 threshold)
        - merror ? Multiclass classification error rate
        - mlogloss ? Multiclass logloss
        - auc: Area under the curve
3. seed [default=0]
    - The random number seed.
    - Can be used for generating reproducible results and also for parameter tuning.
    
Let's start with a set of parameter to train the model. We will come back to tune the parameters later.

In [ ]:
params={
          'learning_rate': 0.1,
          'colsample_bytree' : 0.3,
          'max_depth': 5,
          'num_parallel_tree':1,
          'objective': 'binary:logistic',
          'alpha' : 10,
          'silent': True,
          'tree_method':'gpu_hist',
          'eval_metric':'auc'}
## Train the model
trained_model = xgb.train(
                        params,
                        dtrain,
                        num_boost_round=100, evals=[(dtrain, 'train'), (dval, 'val')])

## Predict the model
prediction = trained_model.predict(dval)

As we can see, the AUC score for validation dataset is 0.89, which is pretty good.

The feature importance score is reported in the XGBoost model. This information can be used to find out the importances of features. There are 3 types:

- "weight" is the number of times a feature appears in a tree
- "gain" is the average gain of splits which use the feature
- "cover" is the average coverage of splits which use the feature

Following is a plot of `gain` importance score, which matches the fact that there are only two informative features in the generated dataset.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
ax = xgb.plot_importance(trained_model, height=0.8, importance_type='gain', max_num_features=n_cols)
ax.grid(False, axis="y")
ax.set_title('Estimated feature importance')
plt.show()

The numerical values of the importance score can be fetched by `get_score` method of the trained model.

In [ ]:
trained_model.get_score(importance_type='gain')

## Exercise
Try to adjust the `num_boost_round` to see the effect in training

XGBoost also provides a Scikit-learn wrapper interface for XGBoost model, which makes it easy to use the tools inside Scikit-learn library. Let's train it with Scikit-learning API. 

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(**params)
model.fit(train_df[list(range(n_cols))].values.get(), train_df['label'].values.get())

Check the model prediction vs the previous prediction results

In [ ]:
np.abs(model.predict(val_df[list(range(n_cols))].values.get()) - (prediction >= 0.5).astype('int')).sum()

Since XGBClassifier is Scikit-learn compatible, we can easily do the hyper-parameters tuning by using `GridSearchCV`. E.g,  we would like to find out the best parameters for `learning_rate`, `max_depth`, `number of estimators`.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
skf = StratifiedKFold(n_splits=4, shuffle = True, random_state = 1001)

params={
          'learning_rate': [0.2, 0.3, 0.4],
          'colsample_bytree' : [0.3],
          'max_depth': [5, 6, 7],
          'objective': ['binary:logistic'],
          'n_estimators':[100, 200],
          'alpha' : [10],
          'silent': [True],
          'tree_method':['gpu_hist'],
          'eval_metric':['auc']}

grid_search = GridSearchCV(model, params, scoring='roc_auc', n_jobs=1, cv=skf, verbose=3)

grid_search.fit(train_df[list(range(n_cols))].values.get(), train_df['label'].values.get())

After the grid search, the best estimator is recorded at `grid_search.best_estimator_` with the best parameter set `grid_search.best_params_`

In [ ]:
grid_search.best_params_

Let's check the performance on the validation dataset.

In [ ]:
from sklearn import metrics
y_predictions = grid_search.best_estimator_.predict_proba(val_df[list(range(n_cols))].values.get())
fpr, tpr, thresholds = metrics.roc_curve(val_df['label'].values.get(), y_predictions[:,1])

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=3,
        label='ROC Curve (area = {:.2f})'.format(metrics.auc(fpr, tpr)))
ax.plot([0, 1], [0, 1], 'k--', lw=2)
ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    title="ROC Curve",
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
)
ax.legend();
plt.show()

Great! We can get AUC 0.9 after tuning. 

## Exercise

Try to use grid search to tune other hyper-parameters to see whether you can make the AUC result better:

## Forest Inference Library

The open source Forest Inference Library (FIL) in RAPIDS can accelerate GBDT and RF inference with GPUs. Users can train models as usual in XGBoost or LightGBM, save them to disk, then use FIL to reload those models and apply them to new data. Using FIL, a single V100 GPU can deliver up to 35x more inference throughput than a CPU-only node with 40 cores. For more details, please refer to this [blog](https://medium.com/rapids-ai/rapids-forest-inference-library-prediction-at-100-million-rows-per-second-19558890bc35).

The XGBoost model can be serialized to disk by `save_model` command:

In [ ]:
grid_search.best_estimator_.save_model('best.model')

It is very easy to use Forest Inference Library. Simply load the model and make prediction. 

In [ ]:
from cuml import ForestInference
fm = ForestInference.load('best.model', output_class=True)

fil_preds_gpu = fm.predict(val_df[list(range(n_cols))])
fil_preds_gpu

## Multiple GPU XGBoost

Single GPU can only fit the size of the data that the memory allows. To train a large dataset, distributed XGBoost training is the solution. Luckily, dask XGBoost integration make distributed XGBoost training an easy task to do.

### Create a Cluster and Client
Let's start by creating a local cluster of workers and a client to interact with that cluster.

In [ ]:
from dask.distributed import Client
from dask_cuda import LocalCUDACluster
import dask_cudf
import dask
# create a local CUDA cluster
cluster = LocalCUDACluster()
client = Client(cluster)
client

We can generate the synthetic distributed dataset by `dask_ml.datasets.make_classification_df` method. It creates `dask_cudf` dataframe with specified chunks. Dask can automatically breakdown the computation into small tasks and schedule the computation in a cluster of GPUs.  

In [ ]:
import dask_ml.datasets
import dask_xgboost
length = 10000
train_frac = 0.8
chunks = 100
validation_chunks = int(train_frac * chunks)
df = dask_ml.datasets.make_classification_df(n_samples=length, chunks=chunks)
df

We can see the `dask_cudf` dataframe has 100 partitions that will be distributed in computation. The dataframe numbers are empty due to the Dask laze computation design. We can split the dask_cudf dataframe into train and validation datasets at the level of partitions. Note, we use the dask low level API `to_delayed` to get the delayed objects

In [ ]:
delayed_x_chunks = df[0].to_delayed()
delayed_y_chunks = df[1].to_delayed()
train_x_dask_df = dask_cudf.from_delayed(delayed_x_chunks[:validation_chunks])
train_y_dask_df = dask_cudf.from_delayed(delayed_y_chunks[:validation_chunks])
val_x_dask_df = dask_cudf.from_delayed(delayed_x_chunks[validation_chunks:])
val_y_dask_df = dask_cudf.from_delayed(delayed_y_chunks[validation_chunks:])

In [ ]:
params={
          'learning_rate': 0.1,
          'colsample_bytree' : 0.3,
          'max_depth': 5,
          'num_parallel_tree':1,
          'objective': 'binary:logistic',
          'alpha' : 10,
          'silent': True,
          'tree_method':'gpu_hist',
          'eval_metric':'auc'}
dtrain = xgb.dask.DaskDMatrix(client, train_x_dask_df, train_y_dask_df)
dval = xgb.dask.DaskDMatrix(client, val_x_dask_df, val_y_dask_df)
bst = xgb.dask.train(client, params, dtrain, num_boost_round=100, evals=[(dtrain, 'train'), (dval, 'val')])

Plot the training curves

In [ ]:
plt.plot(bst['history']['train']['auc'], 'b-', label='train auc')
plt.plot(bst['history']['val']['auc'], 'g-', label='val auc')
plt.legend()
plt.ylabel('auc')
plt.xlabel('round')

We can check the ROC curve for both train the val dataset.

In [ ]:
%matplotlib inline
from sklearn import metrics
import matplotlib.pyplot as plt
y_predictions = xgb.dask.predict(client, bst, dtrain)
fpr, tpr, thresholds = metrics.roc_curve(train_y_dask_df.compute(), y_predictions.compute())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=3,
        label='ROC Curve (area = {:.2f})'.format(metrics.auc(fpr, tpr)))
ax.plot([0, 1], [0, 1], 'k--', lw=2)
ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    title="ROC Curve",
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
)
ax.legend();
plt.show()



In [ ]:
y_predictions = xgb.dask.predict(client, bst, dval)
fpr, tpr, thresholds = metrics.roc_curve(val_y_dask_df.compute(), y_predictions.compute())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=3,
        label='ROC Curve (area = {:.2f})'.format(metrics.auc(fpr, tpr)))
ax.plot([0, 1], [0, 1], 'k--', lw=2)
ax.set(
    xlim=(0, 1),
    ylim=(0, 1),
    title="ROC Curve",
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
)
ax.legend();
plt.show()

We can plot the feature importance score like the single GPU example above

In [ ]:
ax = xgb.plot_importance(bst['booster'], height=0.8, max_num_features=20, importance_type='gain')
ax.grid(False, axis="y")
ax.set_title('Estimated feature importance')
plt.show()

## Homework

XGBoost works for both classification problem and regression problem. Change the target variable to continuous variable and try to follow the above steps but work on a regression problem.